# 08 — Utility files

`backend/app/ml/utils.py` -- shared, side-effect-free helpers used across the whole ML pipeline (training, evaluation, inference, and this notebook's own path-fix cell above). Extracted from the "Utility Functions" section of the original research notebook, kept dependency-light: no FastAPI, no notebook globals, no implicit device or seed state.

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from wherever this notebook actually lives to find the real
    project root (the folder containing both backend/app/ and data/), so every
    relative path used below resolves correctly regardless of which folder
    this notebook is opened from."""
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the Baseera project root (a folder containing both "
        "backend/app/ and data/) above this notebook's location."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


In [ ]:
import inspect
from app.ml import utils

funcs = [name for name, obj in inspect.getmembers(utils, inspect.isfunction) if obj.__module__ == utils.__name__]
classes = [name for name, obj in inspect.getmembers(utils, inspect.isclass) if obj.__module__ == utils.__name__]
print("Functions:", funcs)
print("Classes:  ", classes)


## Reproducibility — `set_seed` / `get_device`

Seed **42** everywhere: dataset split, class-weight computation, K-Means, PyTorch/NumPy/Python RNG.

In [ ]:
from app.ml.utils import set_seed, get_device, stable_text_hash, normalize_text_for_hash

set_seed(42)
device = get_device()
print("Device:", device)

text = "  The Product Was GREAT!!  "
print("normalize_text_for_hash:", repr(normalize_text_for_hash(text)))
print("stable_text_hash:       ", stable_text_hash(text))


## Other utilities

- `checkpoint_fingerprint(model_dir)` -- streamed SHA-256 of model weight files (1MB chunks, never loads the whole ~670MB BERT checkpoint into RAM at once; the fix for a real production OOM incident, see `PROJECT_JOURNEY.md`).
- `optimize_dtypes(df)` -- float64->float32 / int64->smaller-int downcasting where safe.
- `JSONSafeEncoder` / `to_json_safe` / `write_json` -- consistent, schema-versioned JSON output for every file under `results/`.
- `state_code_to_name` / `state_name_to_region` -- Brazilian state lookups used by the geography analytics endpoints.

In [ ]:
from app.ml.utils import state_code_to_name, state_name_to_region

for code_ in ["SP", "RJ", "AM"]:
    name = state_code_to_name(code_)
    print(f"{code_} -> {name} -> region: {state_name_to_region(name)}")


## Also part of "utility files" (configuration & cross-cutting concerns)

`backend/app/core/config.py` (settings), `backend/app/core/logging.py` (structured logging), `backend/app/core/exceptions.py` (typed app errors), `backend/app/core/security.py`, `backend/app/core/rate_limit.py` -- all pure, dependency-light modules imported across the backend, not tied to any single endpoint.